<a href="https://colab.research.google.com/github/upgrade-projects/pysparks-projects/blob/main/Chipotle_Orders.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Chipotle Orders**

You are provided with a dataset titled ‘Chipotle Orders Dataset’, which captures detailed information about individual food orders at Chipotle, including order ID, quantity, item name, item price and customisations (choice descriptions).


With the given [dataset](https://drive.google.com/file/d/1ZsVFwYnJe6pr64VsleeoSgTQph9t8pN9/view?usp=drive_link), solve the following tasks:










In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, count, avg, regexp_replace, desc, split
from pyspark.sql.types import DoubleType

# Initialize SparkSession
spark = SparkSession.builder.appName("ChipotleAnalysis").getOrCreate()

The next step is to load the dataset and perform initial data cleaning. The `item_price` column needs to be converted from a string (e.g., '$2.39') to a numeric type to enable mathematical operations.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Load the dataset
file_path = "/content/drive/MyDrive/Colab_Notebooks/Upgrad_Training/data/chipotle.tsv"
df = spark.read.csv(file_path, sep='\t', header=True, inferSchema=True)

# Display schema and first few rows to understand the data
print("Original Schema:")
df.printSchema()
df.show(5)

# Clean 'item_price' column: remove '$' and cast to DoubleType
df = df.withColumn("item_price_numeric", regexp_replace(col("item_price"), "\\$", "").cast(DoubleType()))

print("\nSchema after cleaning 'item_price':")
df.printSchema()
df.show(5)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Original Schema:
root
 |-- order_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- item_name: string (nullable = true)
 |-- choice_description: string (nullable = true)
 |-- item_price: string (nullable = true)

+--------+--------+--------------------+--------------------+----------+
|order_id|quantity|           item_name|  choice_description|item_price|
+--------+--------+--------------------+--------------------+----------+
|       1|       1|Chips and Fresh T...|                NULL|    $2.39 |
|       1|       1|                Izze|        [Clementine]|    $3.39 |
|       1|       1|    Nantucket Nectar|             [Apple]|    $3.39 |
|       1|       1|Chips and Tomatil...|                NULL|    $2.39 |
|       2|       2|        Chicken Bowl|[Tomatillo-Red Ch...|   $16.98 |
+--------+--------+--------------------+---------

**Task 1: Write a function that returns the total revenue generated from all orders. Hint: Multiply quantity by item_price for each row and then add the results**

In [ ]:
# Task 1: Write a function that returns the total revenue generated from all orders.
def get_total_revenue(dataframe):
    total_revenue = dataframe.withColumn("revenue", col("quantity") * col("item_price_numeric")).select(sum("revenue")).collect()[0][0]
    return total_revenue

# Execute Task 1
total_revenue = get_total_revenue(df)
print(f"Total Revenue Generated: ${total_revenue:.2f}")

Total Revenue Generated: $39237.02


**Task 2: Write a function that returns the most frequently ordered item. Hint: Group by item_name and sum the quantity.**

In [ ]:
# Task 2: Write a function that returns the most frequently ordered item.
def get_most_frequently_ordered_item(dataframe):
    most_frequent_item = dataframe.groupBy("item_name").agg(sum("quantity").alias("total_quantity_ordered")).orderBy(desc("total_quantity_ordered")).first()
    return most_frequent_item['item_name'] if most_frequent_item else None

# Execute Task 2
most_frequent = get_most_frequently_ordered_item(df)
print(f"Most Frequently Ordered Item: {most_frequent}")

Most Frequently Ordered Item: Chicken Bowl


**Task 3: Write a function that returns the total number of unique items sold. Hint: Count the number of distinct values in the item_name column.**

In [ ]:
# Task 3: Write a function that returns the total number of unique items sold.
def get_total_unique_items(dataframe):
    unique_items_count = dataframe.select("item_name").distinct().count()
    return unique_items_count

# Execute Task 3
unique_items_count = get_total_unique_items(df)
print(f"Total Number of Unique Items Sold: {unique_items_count}")

Total Number of Unique Items Sold: 50


**Task 4: Write a function that returns the top 5 items by total revenue. Hint: Multiply quantity and item_price, group by item_name, and sort in descending order.**

In [ ]:
# Task 4: Write a function that returns the top 5 items by total revenue.
def get_top_5_items_by_revenue(dataframe):
    top_5_items = dataframe.withColumn("item_revenue", col("quantity") * col("item_price_numeric")).groupBy("item_name").agg(sum("item_revenue").alias("total_item_revenue")).orderBy(desc("total_item_revenue")).limit(5)
    return top_5_items

# Execute Task 4
top_5_items_df = get_top_5_items_by_revenue(df)
print("\nTop 5 Items by Total Revenue:")
top_5_items_df.show(truncate=False)


Top 5 Items by Total Revenue:
+-------------------+------------------+
|item_name          |total_item_revenue|
+-------------------+------------------+
|Chicken Bowl       |8044.629999999975 |
|Chicken Burrito    |6387.059999999984 |
|Steak Burrito      |4236.12999999999  |
|Steak Bowl         |2479.8099999999995|
|Chips and Guacamole|2475.6199999999963|
+-------------------+------------------+



**Task 5: Write a function that returns the average number of items per order. Hint: Group by order_id, sum the quantity per order, and calculate the mean.**




In [ ]:
# Task 5: Write a function that returns the average number of items per order.
def get_average_items_per_order(dataframe):
    items_per_order = dataframe.groupBy("order_id").agg(sum("quantity").alias("total_items_in_order"))
    average_items = items_per_order.select(avg("total_items_in_order")).collect()[0][0]
    return average_items


# Execute Task 5
average_items_per_order = get_average_items_per_order(df)
print(f"Average Number of Items per Order: {average_items_per_order:.2f}")


Average Number of Items per Order: 2.71


**Task 6: List all beverages sold and their total revenue.**

In [ ]:

# Task 6: List all beverages sold and their total revenue.
# Assuming 'beverage' can be identified from 'item_name' or 'choice_description'
def get_beverages_revenue(dataframe):
    # A simple approach to identify beverages: check for 'drink', 'soda', 'coke', 'tea', etc.
    # This might need refinement based on actual data if categories are not explicit
    beverage_keywords = ["coke", "sprite", "diet coke", "dr. pepper", "mountain dew", "tea", "lemonade", "water", "fountain drink", "iced coffee"]
    # Lowercasing item_name and choice_description for broader matching
    beverages_df = dataframe.filter(
        col("item_name").like("%coke%") |
        col("item_name").like("%sprite%") |
        col("item_name").like("%diet coke%") |
        col("item_name").like("%dr. pepper%") |
        col("item_name").like("%mountain dew%") |
        col("item_name").like("%tea%") |
        col("item_name").like("%lemonade%") |
        col("item_name").like("%water%") |
        col("item_name").like("%fountain drink%") |
        col("item_name").like("%iced coffee%") |
        (col("choice_description").isNotNull() & col("choice_description").like("%coke%")) |
        (col("choice_description").isNotNull() & col("choice_description").like("%sprite%")) |
        (col("choice_description").isNotNull() & col("choice_description").like("%diet coke%")) |
        (col("choice_description").isNotNull() & col("choice_description").like("%dr. pepper%")) |
        (col("choice_description").isNotNull() & col("choice_description").like("%mountain dew%")) |
        (col("choice_description").isNotNull() & col("choice_description").like("%tea%")) |
        (col("choice_description").isNotNull() & col("choice_description").like("%lemonade%")) |
        (col("choice_description").isNotNull() & col("choice_description").like("%water%")) |
        (col("choice_description").isNotNull() & col("choice_description").like("%fountain drink%")) |
        (col("choice_description").isNotNull() & col("choice_description").like("%iced coffee%"))
    ).withColumn("beverage_revenue", col("quantity") * col("item_price_numeric"))

    beverages_total_revenue = beverages_df.groupBy("item_name").agg(sum("beverage_revenue").alias("total_revenue")).orderBy(desc("total_revenue"))
    return beverages_total_revenue


# Execute Task 6
beverages_revenue_df = get_beverages_revenue(df)
print("\nBeverages Sold and their Total Revenue:")
beverages_revenue_df.show(truncate=False)


Beverages Sold and their Total Revenue:
+------------------+------------------+
|item_name         |total_revenue     |
+------------------+------------------+
|Steak Burrito     |4236.12999999999  |
|Steak Bowl        |2479.8099999999995|
|Steak Soft Tacos  |554.5500000000001 |
|Steak Salad Bowl  |391.1499999999997 |
|Steak Crispy Tacos|375.32000000000005|
|Steak Salad       |35.66             |
|Canned Soft Drink |22.5              |
|Burrito           |14.8              |
|6 Pack Soft Drink |12.98             |
|Bowl              |7.4               |
|Crispy Tacos      |7.4               |
+------------------+------------------+



Now, let's call each function and display the results.

In [ ]:
# Stop SparkSession
spark.stop()